# Emotion Classification with Attention on GoEmotions

This notebook builds an end-to-end 6-class Ekman emotion classifier (`joy`, `sadness`, `anger`, `fear`, `surprise`, `disgust`) on the **GoEmotions** dataset. We compare four architectures:
1. **LSTM** (GloVe 100d embeddings + final hidden state)
2. **GRU** (GloVe 100d embeddings + final hidden state)
3. **BiLSTM + Attention** (GloVe 100d + additive Bahdanau attention over packed states)
4. **Fine-tuned DistilBERT** (`distilbert-base-uncased` via HuggingFace `Trainer`)

All models use inverse-frequency class weights to handle severe class imbalance and select checkpoints strictly by **validation macro F1**. At the end, the best model plus the attention model and a ready-to-run **Streamlit** app are packaged into a single downloadable deployment archive.

> **Note:** After running the installation cell below, if an import error appears, **restart the Colab runtime session** (`Runtime -> Restart session`) and run all cells again.

In [ ]:
!pip install -q gensim accelerate

In [ ]:
import os
import re
import json
import time
import copy
import html
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# ---------------- GLOBAL FLAGS & CONFIG ----------------
QUICK_MODE = False          # True = small subset + fewer epochs for smoke test; False = full dataset
FREEZE_EMBEDDINGS = True    # Freeze pre-trained GloVe embeddings during RNN training

CONFIG = {
    "seed": 42,
    "max_len": 64,
    "batch_size": 64,
    "emb_dim": 100,
    "hidden_size": 128,
    "dropout": 0.3,
    "glove_name": "glove-wiki-gigaword-100",
    "freeze_embeddings": FREEZE_EMBEDDINGS,
    "rnn_lr": 1e-3,
    "rnn_epochs": 3 if QUICK_MODE else 10,
    "patience": 2,
    "grad_clip": 1.0,
    "transformer_name": "distilbert-base-uncased",
    "transformer_lr": 2e-5,
    "transformer_batch_size": 32,
    "transformer_epochs": 1 if QUICK_MODE else 3,
    "quick_train_size": 2500,
    "quick_eval_size": 500,
}

def set_all_seeds(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    try:
        from transformers import set_seed
        set_seed(seed)
    except Exception:
        pass

set_all_seeds(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

print(f"Device: {device} | QUICK_MODE: {QUICK_MODE}")
print("Configuration:", json.dumps(CONFIG, indent=2))

## Stage 2: Load & Explore GoEmotions

We load the `simplified` configuration of GoEmotions, inspect the 28 fine-grained label names directly from the dataset features, and map them to the 6 Ekman emotions (`neutral` is excluded). We keep only examples that map to **exactly one** Ekman class, dropping neutral-only and multi-class ambiguous examples.

In [ ]:
def load_go_emotions():
    errors = []
    for repo in ["google-research-datasets/go_emotions", "go_emotions"]:
        try:
            print(f"Trying to load '{repo}' (config='simplified')...")
            ds = load_dataset(repo, "simplified")
            print(f"Successfully loaded '{repo}'.")
            return ds
        except Exception as e:
            errors.append(f"{repo}: {e}")
    raise RuntimeError("Failed to load GoEmotions dataset:\n" + "\n".join(errors))

raw_datasets = load_go_emotions()

# Read 28 label names dynamically from dataset features
fine_label_names = raw_datasets["train"].features["labels"].feature.names
print(f"Total fine-grained labels ({len(fine_label_names)}): {fine_label_names}")

EKMAN_GROUPS = {
    "anger": ["anger", "annoyance", "disapproval"],
    "disgust": ["disgust"],
    "fear": ["fear", "nervousness"],
    "joy": [
        "joy", "amusement", "approval", "excitement", "gratitude", "love",
        "optimism", "relief", "pride", "admiration", "desire", "caring",
    ],
    "sadness": ["sadness", "disappointment", "embarrassment", "grief", "remorse"],
    "surprise": ["surprise", "realization", "confusion", "curiosity"],
}

EMOTIONS = ["joy", "sadness", "anger", "fear", "surprise", "disgust"]
label2id = {emo: i for i, emo in enumerate(EMOTIONS)}
id2label = {i: emo for i, emo in enumerate(EMOTIONS)}

# Map fine-grained integer index -> Ekman emotion string (neutral maps to None)
fine_id_to_ekman = {}
for idx, name in enumerate(fine_label_names):
    mapped = None
    for ekman_emo, fine_list in EKMAN_GROUPS.items():
        if name in fine_list:
            mapped = ekman_emo
            break
    fine_id_to_ekman[idx] = mapped

def filter_and_map_split(split_ds, split_name: str, max_samples=None):
    kept_texts, kept_labels = [], []
    dropped_neutral, dropped_ambiguous = 0, 0

    for ex in split_ds:
        ekman_set = {
            fine_id_to_ekman[i]
            for i in ex["labels"]
            if fine_id_to_ekman.get(i) is not None
        }
        if len(ekman_set) == 0:
            dropped_neutral += 1
        elif len(ekman_set) > 1:
            dropped_ambiguous += 1
        else:
            only_emo = next(iter(ekman_set))
            kept_texts.append(ex["text"])
            kept_labels.append(label2id[only_emo])

    total = len(split_ds)
    kept = len(kept_texts)
    print(
        f"[{split_name:10s}] total={total:6d} | kept={kept:6d} ({kept/total*100:.1f}%) | "
        f"dropped_neutral={dropped_neutral:5d} | dropped_ambiguous={dropped_ambiguous:5d}"
    )

    if max_samples is not None and kept > max_samples:
        rng = np.random.default_rng(CONFIG["seed"])
        indices = rng.choice(kept, size=max_samples, replace=False)
        kept_texts = [kept_texts[i] for i in indices]
        kept_labels = [kept_labels[i] for i in indices]
        print(f"  -> QUICK_MODE subsampled {split_name} to {len(kept_texts)} examples.")

    return {"text": kept_texts, "label": kept_labels}

train_data = filter_and_map_split(
    raw_datasets["train"],
    "train",
    CONFIG["quick_train_size"] if QUICK_MODE else None,
)
val_data = filter_and_map_split(
    raw_datasets["validation"],
    "validation",
    CONFIG["quick_eval_size"] if QUICK_MODE else None,
)
test_data = filter_and_map_split(
    raw_datasets["test"],
    "test",
    CONFIG["quick_eval_size"] if QUICK_MODE else None,
)

In [ ]:
# Split sizes & class counts
train_counts = Counter(train_data["label"])
val_counts = Counter(val_data["label"])
test_counts = Counter(test_data["label"])

dist_df = pd.DataFrame({
    "Emotion": EMOTIONS,
    "Train": [train_counts[i] for i in range(len(EMOTIONS))],
    "Validation": [val_counts[i] for i in range(len(EMOTIONS))],
    "Test": [test_counts[i] for i in range(len(EMOTIONS))],
})
dist_df["Train %"] = (dist_df["Train"] / len(train_data["label"]) * 100).round(2)
print("\nClass Distribution Across Splits:")
print(dist_df.to_string(index=False))

# Compute inverse-frequency class weights from the training split only: w_c = N / (K * n_c)
num_classes = len(EMOTIONS)
total_train = len(train_data["label"])
class_weights_np = np.array(
    [total_train / (num_classes * max(train_counts[c], 1)) for c in range(num_classes)],
    dtype=np.float32,
)
class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float32, device=device)

print("\nComputed Inverse-Frequency Class Weights (Train Set):")
for emo, w in zip(EMOTIONS, class_weights_np):
    print(f"  {emo:10s}: {w:.4f}")

# Plot class distribution and word length histogram
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = ["#22c55e", "#3b82f6", "#ef4444", "#a855f7", "#f59e0b", "#14b8a6"]
axes[0].bar(EMOTIONS, [train_counts[i] for i in range(num_classes)], color=colors)
axes[0].set_title("Training Set Class Distribution (6 Ekman Classes)")
axes[0].set_ylabel("Number of Examples")
axes[0].grid(axis="y", alpha=0.3)

word_lengths = [len(t.split()) for t in train_data["text"]]
axes[1].hist(word_lengths, bins=30, color="#475569", edgecolor="white")
axes[1].axvline(np.mean(word_lengths), color="#ef4444", linestyle="--", label=f"Mean: {np.mean(word_lengths):.1f}")
axes[1].set_title("Distribution of Sentence Word Lengths (Train)")
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("results/data_exploration.png", dpi=150)
plt.show()

# Print sample sentences per emotion class
print("\nSample Sentences per Emotion Class:")
for cid, emo in enumerate(EMOTIONS):
    samples = [t for t, y in zip(train_data["text"], train_data["label"]) if y == cid][:2]
    print(f"  [{emo.upper()}]")
    for s in samples:
        print(f"    - {s}")

## Stage 3 & 5: Shared Module (`emotion_inference.py`)

We create `emotion_inference.py` using `%%writefile` **before** building the vocabulary or training models so that both notebook training and the exported Streamlit application import the exact same `clean_tokenize`, `SeqClassifier`, and `load_predictor` definitions.

In [ ]:
%%writefile emotion_inference.py
import json
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import numpy as np
import torch
import torch.nn as nn

EMOTIONS: List[str] = ["joy", "sadness", "anger", "fear", "surprise", "disgust"]

TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?|[!?.,]")

def clean_tokenize(text: str) -> List[str]:
    """Lowercase, normalize GoEmotions placeholders, and keep apostrophe words + !?.,"""
    text = text.lower()
    text = re.sub(r"\[name\]", "name", text)
    text = re.sub(r"\[religion\]", "religion", text)
    return TOKEN_PATTERN.findall(text)


class AdditiveAttention(nn.Module):
    """Bahdanau-style additive attention: score_t = v^T tanh(W h_t) with masked softmax."""

    def __init__(self, hidden_dim: int):
        super().__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden_states: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # hidden_states: (B, L, H), mask: (B, L) boolean where True = valid token
        energy = torch.tanh(self.W(hidden_states))          # (B, L, H)
        scores = self.v(energy).squeeze(-1)                 # (B, L)
        scores = scores.masked_fill(~mask, -1e9)
        attn_weights = torch.softmax(scores, dim=-1)        # (B, L)
        context = torch.bmm(attn_weights.unsqueeze(1), hidden_states).squeeze(1)  # (B, H)
        return context, attn_weights


class SeqClassifier(nn.Module):
    """Unified LSTM / GRU sequence classifier with optional bidirectionality and additive attention."""

    def __init__(
        self,
        vocab_size: int,
        emb_dim: int = 100,
        hidden_size: int = 128,
        num_classes: int = 6,
        dropout: float = 0.3,
        rnn_type: str = "lstm",
        bidirectional: bool = False,
        use_attention: bool = False,
        pretrained_embeddings: Optional[torch.Tensor] = None,
        freeze_embeddings: bool = True,
        pad_idx: int = 0,
    ):
        super().__init__()
        self.rnn_type = rnn_type.lower()
        self.bidirectional = bidirectional
        self.use_attention = use_attention
        self.pad_idx = pad_idx

        if pretrained_embeddings is not None:
            self.embedding = nn.Embedding.from_pretrained(
                pretrained_embeddings.clone().detach().float(),
                freeze=freeze_embeddings,
                padding_idx=pad_idx,
            )
        else:
            self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)

        self.dropout = nn.Dropout(dropout)
        rnn_cls = nn.LSTM if self.rnn_type == "lstm" else nn.GRU
        self.rnn = rnn_cls(
            input_size=emb_dim,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=bidirectional,
        )

        out_dim = hidden_size * (2 if bidirectional else 1)
        self.attention = AdditiveAttention(out_dim) if use_attention else None
        self.fc = nn.Linear(out_dim, num_classes)

    def forward(
        self, words: torch.Tensor, lengths: torch.Tensor
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        # words: (B, L), lengths: (B,)
        B, L = words.shape
        emb = self.dropout(self.embedding(words))  # (B, L, E)

        clamped_lengths = lengths.clamp(min=1, max=L).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, clamped_lengths, batch_first=True, enforce_sorted=False
        )
        packed_out, hidden = self.rnn(packed)
        rnn_out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=L
        )  # (B, L, H_out)

        attn_weights = None
        if self.use_attention and self.attention is not None:
            device = words.device
            arange = torch.arange(L, device=device).unsqueeze(0)
            mask = arange < lengths.to(device).unsqueeze(1)  # (B, L)
            rep, attn_weights = self.attention(rnn_out, mask)
        else:
            h_n = hidden[0] if self.rnn_type == "lstm" else hidden
            if self.bidirectional:
                rep = torch.cat([h_n[-2], h_n[-1]], dim=-1)
            else:
                rep = h_n[-1]

        logits = self.fc(self.dropout(rep))
        return logits, attn_weights


def _encode_tokens(tokens: List[str], vocab: Dict[str, int], max_len: int) -> Tuple[torch.Tensor, torch.Tensor, List[str]]:
    truncated = tokens[:max_len] if tokens else ["<UNK>"]
    length = len(truncated)
    unk_idx = vocab.get("<UNK>", 1)
    pad_idx = vocab.get("<PAD>", 0)
    ids = [vocab.get(tok, unk_idx) for tok in truncated]
    if len(ids) < max_len:
        ids = ids + [pad_idx] * (max_len - len(ids))
    words_t = torch.tensor([ids], dtype=torch.long)
    lengths_t = torch.tensor([length], dtype=torch.long)
    return words_t, lengths_t, truncated


def _load_rnn_bundle(folder: Path, rnn_cfg: Dict, num_labels: int, device: torch.device):
    vocab_path = folder / "vocab.json"
    weights_path = folder / "model.pt"
    with open(vocab_path, "r", encoding="utf-8") as f:
        vocab = json.load(f)

    model = SeqClassifier(
        vocab_size=len(vocab),
        emb_dim=rnn_cfg.get("emb_dim", 100),
        hidden_size=rnn_cfg.get("hidden_size", 128),
        num_classes=num_labels,
        dropout=rnn_cfg.get("dropout", 0.3),
        rnn_type=rnn_cfg.get("rnn_type", "lstm"),
        bidirectional=rnn_cfg.get("bidirectional", True),
        use_attention=rnn_cfg.get("use_attention", True),
    )
    state = torch.load(weights_path, map_location=device, weights_only=True)
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    return model, vocab


def load_predictor(root: Union[str, Path] = "."):
    """
    Loads the best classifier and the BiLSTM+Attention explainer from root.
    Returns (predict_fn, deployment_config).
    """
    root_path = Path(root)
    cfg_path = root_path / "deployment_config.json"
    if not cfg_path.exists():
        raise FileNotFoundError(f"Missing deployment_config.json in {root_path}")

    with open(cfg_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    labels: List[str] = cfg.get("labels", EMOTIONS)
    model_type: str = cfg.get("model_type", "rnn")
    rnn_cfg: Dict = cfg.get("rnn", {})
    max_len: int = int(rnn_cfg.get("max_len", 64))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Load primary prediction model (either Transformer or RNN)
    primary_rnn = None
    primary_vocab = None
    hf_model = None
    hf_tokenizer = None

    if model_type == "transformer":
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        hub_repo = cfg.get("hub_repo")
        model_source = hub_repo if hub_repo else str(root_path / "model")
        hf_tokenizer = AutoTokenizer.from_pretrained(model_source)
        hf_model = AutoModelForSequenceClassification.from_pretrained(model_source)
        hf_model.to(device)
        hf_model.eval()
    else:
        primary_rnn, primary_vocab = _load_rnn_bundle(
            root_path / "model", rnn_cfg, len(labels), device
        )

    # 2. Load BiLSTM+Attention model for token-level attention weights
    attn_rnn = None
    attn_vocab = None
    attn_dir = root_path / "attention_model"
    attn_cfg = {
        "emb_dim": rnn_cfg.get("emb_dim", 100),
        "hidden_size": rnn_cfg.get("hidden_size", 128),
        "dropout": rnn_cfg.get("dropout", 0.3),
        "rnn_type": "lstm",
        "bidirectional": True,
        "use_attention": True,
    }
    if primary_rnn is not None and rnn_cfg.get("use_attention", False):
        attn_rnn, attn_vocab = primary_rnn, primary_vocab
    elif (attn_dir / "model.pt").exists() and (attn_dir / "vocab.json").exists():
        attn_rnn, attn_vocab = _load_rnn_bundle(attn_dir, attn_cfg, len(labels), device)

    def predict(text: str) -> Dict:
        raw_tokens = clean_tokenize(text)
        if not raw_tokens:
            raw_tokens = ["<UNK>"]

        # Compute attention weights from BiLSTM+Attention
        tokens_out = raw_tokens[:max_len]
        attn_list: Optional[List[float]] = None
        if attn_rnn is not None and attn_vocab is not None:
            w_t, l_t, tokens_out = _encode_tokens(raw_tokens, attn_vocab, max_len)
            with torch.no_grad():
                logits_attn, attn_w = attn_rnn(w_t.to(device), l_t.to(device))
                if attn_w is not None:
                    valid_len = int(l_t.item())
                    attn_list = [float(x) for x in attn_w[0, :valid_len].cpu().tolist()]

        # Compute class probabilities from best model
        if model_type == "transformer" and hf_model is not None and hf_tokenizer is not None:
            inputs = hf_tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=max_len,
                padding=True,
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                logits = hf_model(**inputs).logits
                probs_t = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        else:
            w_t, l_t, tokens_out = _encode_tokens(raw_tokens, primary_vocab, max_len)
            with torch.no_grad():
                logits, _ = primary_rnn(w_t.to(device), l_t.to(device))
                probs_t = torch.softmax(logits, dim=-1)[0].cpu().numpy()

        probs_dict = {labels[i]: float(probs_t[i]) for i in range(len(labels))}
        best_idx = int(np.argmax(probs_t))
        return {
            "label": labels[best_idx],
            "probs": probs_dict,
            "tokens": tokens_out,
            "attention": attn_list,
        }

    return predict, cfg

## Stage 3 & 4: Clean, Tokenize, Pad & Load GloVe Embeddings

We import `clean_tokenize` and `SeqClassifier` from `emotion_inference.py`, load `glove-wiki-gigaword-100` via `gensim.downloader` (with a safe random fallback if offline), build the vocabulary from training tokens (`min_freq=2`) plus validation/test tokens present in GloVe, and construct fixed-length (`max_len=64`) DataLoaders.

In [ ]:
from emotion_inference import EMOTIONS, clean_tokenize, SeqClassifier, load_predictor
import gensim.downloader as api

# 1. Load pre-trained GloVe 100d vectors with fallback
glove_vectors = None
try:
    print(f"Loading pre-trained '{CONFIG['glove_name']}' via gensim.downloader...")
    glove_vectors = api.load(CONFIG["glove_name"])
    print(f"Loaded GloVe with {len(glove_vectors):,} word vectors.")
except Exception as e:
    print(f"WARNING: Could not load GloVe ({e}). Falling back to random normal embeddings.")

# 2. Tokenize all splits
train_tokens = [clean_tokenize(t) for t in train_data["text"]]
val_tokens = [clean_tokenize(t) for t in val_data["text"]]
test_tokens = [clean_tokenize(t) for t in test_data["text"]]

# 3. Build vocabulary: train words (min_freq=2) + val/test words that exist in GloVe
word_freq = Counter(tok for seq in train_tokens for tok in seq)
vocab = {"<PAD>": 0, "<UNK>": 1}
for word, freq in word_freq.items():
    if freq >= 2 and word not in vocab:
        vocab[word] = len(vocab)

extra_glove_words = 0
if glove_vectors is not None:
    for seq in val_tokens + test_tokens:
        for word in seq:
            if word not in vocab and word in glove_vectors:
                vocab[word] = len(vocab)
                extra_glove_words += 1

print(f"Vocabulary size: {len(vocab):,} (including {extra_glove_words:,} val/test words found in GloVe)")

# 4. Build embedding matrix (zeros for <PAD> and words missing in GloVe)
emb_dim = CONFIG["emb_dim"]
embedding_matrix = np.zeros((len(vocab), emb_dim), dtype=np.float32)
hits = 0

for word, idx in vocab.items():
    if idx == 0:  # <PAD>
        continue
    if glove_vectors is not None and word in glove_vectors:
        embedding_matrix[idx] = glove_vectors[word]
        hits += 1
    elif glove_vectors is None:
        rng = np.random.default_rng(CONFIG["seed"] + idx)
        embedding_matrix[idx] = rng.normal(0.0, 0.1, size=(emb_dim,)).astype(np.float32)

coverage = hits / max(len(vocab) - 1, 1) * 100
print(f"GloVe vocabulary coverage: {hits:,} / {len(vocab)-1:,} words ({coverage:.2f}%)")
embedding_tensor = torch.tensor(embedding_matrix, dtype=torch.float32)

# Save vocabulary for deployment
with open("models/vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f)

# 5. Encode and pad/truncate every sentence to fixed max_len=64
class EmotionSequenceDataset(Dataset):
    def __init__(self, token_lists, labels, vocab, max_len=64):
        self.words = []
        self.lengths = []
        self.labels = list(labels)
        pad_id = vocab["<PAD>"]
        unk_id = vocab["<UNK>"]

        for seq in token_lists:
            trunc = seq[:max_len] if len(seq) > 0 else ["<UNK>"]
            true_len = len(trunc)
            ids = [vocab.get(w, unk_id) for w in trunc]
            if len(ids) < max_len:
                ids = ids + [pad_id] * (max_len - len(ids))
            self.words.append(ids)
            self.lengths.append(true_len)

        self.words = torch.tensor(self.words, dtype=torch.long)
        self.lengths = torch.tensor(self.lengths, dtype=torch.long)
        self.labels = torch.tensor(self.labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.words[idx], self.lengths[idx], self.labels[idx]

train_rnn_ds = EmotionSequenceDataset(train_tokens, train_data["label"], vocab, CONFIG["max_len"])
val_rnn_ds = EmotionSequenceDataset(val_tokens, val_data["label"], vocab, CONFIG["max_len"])
test_rnn_ds = EmotionSequenceDataset(test_tokens, test_data["label"], vocab, CONFIG["max_len"])

train_loader = DataLoader(train_rnn_ds, batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(val_rnn_ds, batch_size=CONFIG["batch_size"], shuffle=False)
test_loader = DataLoader(test_rnn_ds, batch_size=CONFIG["batch_size"], shuffle=False)

print(f"DataLoaders ready | Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

## Stage 6: Train & Evaluate Sequence Models (LSTM, GRU, BiLSTM + Attention)

We use a single generic training loop with **Adam**, **class-weighted CrossEntropyLoss**, gradient clipping, and early stopping on **validation macro F1** (never the test set). Best weights are restored via `copy.deepcopy` and saved to `models/<name>.pt`.

In [ ]:
def evaluate_rnn(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_probs, all_targets = [], [], []

    with torch.no_grad():
        for words, lengths, labels in loader:
            words = words.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)

            logits, _ = model(words, lengths)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)

            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            preds = np.argmax(probs, axis=-1)

            all_probs.append(probs)
            all_preds.append(preds)
            all_targets.append(labels.cpu().numpy())

    all_probs = np.vstack(all_probs)
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    avg_loss = total_loss / max(len(all_targets), 1)

    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(all_targets, all_preds, average="weighted", zero_division=0)
    report = classification_report(
        all_targets, all_preds, target_names=EMOTIONS, output_dict=True, zero_division=0
    )
    cm = confusion_matrix(all_targets, all_preds, labels=list(range(len(EMOTIONS))))

    return {
        "loss": avg_loss,
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "report": report,
        "confusion_matrix": cm,
        "predictions": all_preds,
        "probabilities": all_probs,
    }


def train_rnn_model(name: str, rnn_type: str, bidirectional: bool, use_attention: bool):
    print(f"\n{'='*65}\nTraining {name} (rnn_type={rnn_type}, bi={bidirectional}, attn={use_attention})\n{'='*65}")
    set_all_seeds(CONFIG["seed"])

    model = SeqClassifier(
        vocab_size=len(vocab),
        emb_dim=CONFIG["emb_dim"],
        hidden_size=CONFIG["hidden_size"],
        num_classes=len(EMOTIONS),
        dropout=CONFIG["dropout"],
        rnn_type=rnn_type,
        bidirectional=bidirectional,
        use_attention=use_attention,
        pretrained_embeddings=embedding_tensor,
        freeze_embeddings=CONFIG["freeze_embeddings"],
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=CONFIG["rnn_lr"]
    )

    history = {"train_loss": [], "val_loss": [], "val_macro_f1": []}
    best_val_f1 = -1.0
    best_state = copy.deepcopy(model.state_dict())
    patience_counter = 0
    epochs_run = 0

    start_time = time.time()
    for epoch in range(1, CONFIG["rnn_epochs"] + 1):
        epochs_run = epoch
        model.train()
        running_loss = 0.0
        total_seen = 0

        for words, lengths, labels in train_loader:
            words, lengths, labels = words.to(device), lengths.to(device), labels.to(device)
            optimizer.zero_grad()
            logits, _ = model(words, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            total_seen += labels.size(0)

        train_loss = running_loss / max(total_seen, 1)
        val_metrics = evaluate_rnn(model, val_loader, criterion)
        val_loss = val_metrics["loss"]
        val_f1 = val_metrics["macro_f1"]

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_macro_f1"].append(val_f1)

        print(
            f"Epoch {epoch:02d}/{CONFIG['rnn_epochs']:02d} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro F1: {val_f1:.4f}"
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= CONFIG["patience"]:
                print(f"Early stopping triggered at epoch {epoch} (best Val Macro F1: {best_val_f1:.4f})")
                break

    train_time_sec = time.time() - start_time
    model.load_state_dict(best_state)

    safe_slug = name.lower().replace("+", "plus").replace(" ", "_")
    save_path = f"models/{safe_slug}.pt"
    torch.save(best_state, save_path)
    print(f"Saved best {name} checkpoint to {save_path} | Training time: {train_time_sec:.1f}s ({train_time_sec/60:.2f} min)")

    test_metrics = evaluate_rnn(model, test_loader, criterion)
    test_metrics.update({
        "val_macro_f1": float(best_val_f1),
        "epochs": epochs_run,
        "train_time_sec": float(train_time_sec),
        "train_time_min": float(train_time_sec / 60.0),
        "history": history,
        "model_path": save_path,
        "rnn_config": {
            "rnn_type": rnn_type,
            "bidirectional": bidirectional,
            "use_attention": use_attention,
            "emb_dim": CONFIG["emb_dim"],
            "hidden_size": CONFIG["hidden_size"],
            "max_len": CONFIG["max_len"],
            "dropout": CONFIG["dropout"],
        },
    })
    return model, test_metrics


results = {}
trained_models = {}

rnn_specs = [
    ("LSTM", "lstm", False, False),
    ("GRU", "gru", False, False),
    ("BiLSTM+Attention", "lstm", True, True),
]

for m_name, r_type, bi, attn in rnn_specs:
    model_obj, res_dict = train_rnn_model(m_name, r_type, bi, attn)
    trained_models[m_name] = model_obj
    results[m_name] = res_dict

## Stage 7: Fine-Tune DistilBERT (`distilbert-base-uncased`)

We fine-tune DistilBERT using `AutoTokenizer`, `AutoModelForSequenceClassification`, `DataCollatorWithPadding`, and a custom `WeightedTrainer` overriding `compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None)` so the exact same inverse-frequency class weights are applied.

In [ ]:
from datasets import Dataset as HFDataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

set_all_seeds(CONFIG["seed"])
tokenizer = AutoTokenizer.from_pretrained(CONFIG["transformer_name"])

hf_train = HFDataset.from_dict(train_data)
hf_val = HFDataset.from_dict(val_data)
hf_test = HFDataset.from_dict(test_data)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=CONFIG["max_len"])

hf_train_tok = hf_train.map(tokenize_batch, batched=True, remove_columns=["text"])
hf_val_tok = hf_val.map(tokenize_batch, batched=True, remove_columns=["text"])
hf_test_tok = hf_test.map(tokenize_batch, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

hf_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["transformer_name"],
    num_labels=len(EMOTIONS),
    id2label=id2label,
    label2id=label2id,
).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    return {"accuracy": float(acc), "macro_f1": float(macro_f1)}

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor.to(logits.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="models/distilbert_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    learning_rate=CONFIG["transformer_lr"],
    per_device_train_batch_size=CONFIG["transformer_batch_size"],
    per_device_eval_batch_size=CONFIG["transformer_batch_size"] * 2,
    num_train_epochs=CONFIG["transformer_epochs"],
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=CONFIG["seed"],
    logging_strategy="epoch",
)

trainer = WeightedTrainer(
    model=hf_model,
    args=training_args,
    train_dataset=hf_train_tok,
    eval_dataset=hf_val_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting DistilBERT fine-tuning...")
t0 = time.time()
trainer.train()
distilbert_time_sec = time.time() - t0
print(f"DistilBERT training finished in {distilbert_time_sec:.1f}s ({distilbert_time_sec/60:.2f} min)")

# Save fine-tuned model and tokenizer
distilbert_dir = "models/distilbert-emotion"
trainer.save_model(distilbert_dir)
tokenizer.save_pretrained(distilbert_dir)

# Validation macro F1 of the best loaded checkpoint
val_eval = trainer.evaluate(eval_dataset=hf_val_tok)
best_val_macro_f1 = float(val_eval["eval_macro_f1"])

# Evaluate on test set with the exact same structure as RNN models
test_output = trainer.predict(hf_test_tok)
test_logits = test_output.predictions
test_targets = np.array(test_data["label"])
test_probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()
test_preds = np.argmax(test_probs, axis=-1)

results["DistilBERT"] = {
    "loss": float(test_output.metrics.get("test_loss", 0.0)),
    "accuracy": float(accuracy_score(test_targets, test_preds)),
    "macro_f1": float(f1_score(test_targets, test_preds, average="macro", zero_division=0)),
    "weighted_f1": float(f1_score(test_targets, test_preds, average="weighted", zero_division=0)),
    "val_macro_f1": best_val_macro_f1,
    "epochs": CONFIG["transformer_epochs"],
    "train_time_sec": float(distilbert_time_sec),
    "train_time_min": float(distilbert_time_sec / 60.0),
    "report": classification_report(
        test_targets, test_preds, target_names=EMOTIONS, output_dict=True, zero_division=0
    ),
    "confusion_matrix": confusion_matrix(
        test_targets, test_preds, labels=list(range(len(EMOTIONS)))
    ),
    "predictions": test_preds,
    "probabilities": test_probs,
    "model_path": distilbert_dir,
}
print(f"DistilBERT Test Macro F1: {results['DistilBERT']['macro_f1']:.4f} | Val Macro F1: {best_val_macro_f1:.4f}")

## Stage 8: Comparison & Evaluation

We compare all four architectures across accuracy, macro F1, weighted F1, validation macro F1, per-class Precision/Recall/F1, training curves, and normalized confusion matrices. All tables and figures are saved to `results/`.

In [ ]:
model_names = ["LSTM", "GRU", "BiLSTM+Attention", "DistilBERT"]

# 1. Overall comparison table
overall_rows = []
for m in model_names:
    r = results[m]
    overall_rows.append({
        "Model": m,
        "Accuracy": round(r["accuracy"], 4),
        "Macro_F1": round(r["macro_f1"], 4),
        "Weighted_F1": round(r["weighted_f1"], 4),
        "Best_Val_Macro_F1": round(r["val_macro_f1"], 4),
        "Epochs": r["epochs"],
        "Train_Time_Min": round(r["train_time_min"], 2),
    })

overall_df = pd.DataFrame(overall_rows)
overall_df.to_csv("results/overall_comparison.csv", index=False)
print("=== OVERALL MODEL COMPARISON (TEST SET + BEST VAL MACRO F1) ===")
print(overall_df.to_string(index=False))

# 2. Per-class Precision / Recall / F1 table
per_class_rows = []
for m in model_names:
    rep = results[m]["report"]
    for emo in EMOTIONS:
        per_class_rows.append({
            "Model": m,
            "Emotion": emo,
            "Precision": round(rep[emo]["precision"], 4),
            "Recall": round(rep[emo]["recall"], 4),
            "F1": round(rep[emo]["f1-score"], 4),
            "Support": int(rep[emo]["support"]),
        })

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv("results/per_class_metrics.csv", index=False)
print("\n=== PER-CLASS F1 SCORES BY MODEL ===")
pivot_f1 = per_class_df.pivot(index="Emotion", columns="Model", values="F1").loc[EMOTIONS, model_names]
print(pivot_f1.to_string())

# 3. 2x2 Comparison Charts
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# (0,0) Accuracy vs Macro F1 per model
x = np.arange(len(model_names))
width = 0.36
axes[0, 0].bar(x - width/2, [results[m]["accuracy"] for m in model_names], width, label="Test Accuracy", color="#3b82f6")
axes[0, 0].bar(x + width/2, [results[m]["macro_f1"] for m in model_names], width, label="Test Macro F1", color="#10b981")
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(model_names)
axes[0, 0].set_ylim(0, 1.0)
axes[0, 0].set_title("Test Accuracy vs. Macro F1 per Model")
axes[0, 0].legend()
axes[0, 0].grid(axis="y", alpha=0.3)

# (0,1) Per-class F1 grouped bars
emo_x = np.arange(len(EMOTIONS))
bar_w = 0.2
palette = ["#64748b", "#0ea5e9", "#8b5cf6", "#f59e0b"]
for idx, m in enumerate(model_names):
    f1s = [results[m]["report"][e]["f1-score"] for e in EMOTIONS]
    axes[0, 1].bar(emo_x + (idx - 1.5) * bar_w, f1s, bar_w, label=m, color=palette[idx])
axes[0, 1].set_xticks(emo_x)
axes[0, 1].set_xticklabels(EMOTIONS)
axes[0, 1].set_ylim(0, 1.0)
axes[0, 1].set_title("Per-Class Test F1 Score Across Architectures")
axes[0, 1].legend()
axes[0, 1].grid(axis="y", alpha=0.3)

# (1,0) Validation Macro F1 per epoch for the RNN models
for idx, m in enumerate(["LSTM", "GRU", "BiLSTM+Attention"]):
    hist_f1 = results[m]["history"]["val_macro_f1"]
    axes[1, 0].plot(range(1, len(hist_f1) + 1), hist_f1, marker="o", linewidth=2, label=m)
axes[1, 0].set_title("Validation Macro F1 per Epoch (RNN Architectures)")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Validation Macro F1")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# (1,1) Training time per model (minutes)
times_min = [results[m]["train_time_min"] for m in model_names]
axes[1, 1].barh(model_names, times_min, color=palette)
axes[1, 1].set_xlabel("Training Time (Minutes)")
axes[1, 1].set_title("Training Time per Model")
axes[1, 1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("results/model_comparison_charts.png", dpi=150)
plt.show()

# 4. 2x2 Normalized Confusion Matrices
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for ax, m in zip(axes.flat, model_names):
    cm = results[m]["confusion_matrix"].astype(np.float64)
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums != 0)

    im = ax.imshow(cm_norm, cmap="Blues", vmin=0.0, vmax=1.0)
    ax.set_title(f"{m} (Macro F1: {results[m]['macro_f1']:.3f})")
    ax.set_xticks(range(len(EMOTIONS)))
    ax.set_yticks(range(len(EMOTIONS)))
    ax.set_xticklabels(EMOTIONS, rotation=35, ha="right")
    ax.set_yticklabels(EMOTIONS)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    for i in range(len(EMOTIONS)):
        for j in range(len(EMOTIONS)):
            val = cm_norm[i, j]
            text_color = "white" if val > 0.5 else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=text_color, fontsize=9)

plt.tight_layout()
plt.savefig("results/confusion_matrices.png", dpi=150)
plt.show()

## Stage 9: Attention Interpretation (BiLSTM + Attention)

We inspect how the additive attention mechanism distributes weight across tokens on 8 test examples (6 correctly classified examples across emotions + 2 misclassified examples), aggregate the top-10 most attended words per emotion across the test set, and print a data-driven discussion directly from our computed metrics.

In [ ]:
bilstm_attn_model = trained_models["BiLSTM+Attention"]
bilstm_attn_model.eval()

# Extract token-level attention weights on the entire test set
test_attn_records = []
word_attn_by_true_emotion = {emo: defaultdict(list) for emo in EMOTIONS}

with torch.no_grad():
    for idx in range(len(test_rnn_ds)):
        w_t, l_t, y_t = test_rnn_ds[idx]
        valid_len = int(l_t.item())
        logits, attn_w = bilstm_attn_model(
            w_t.unsqueeze(0).to(device), l_t.unsqueeze(0).to(device)
        )
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        pred_id = int(np.argmax(probs))
        true_id = int(y_t.item())
        conf = float(probs[pred_id])
        weights = attn_w[0, :valid_len].cpu().numpy()
        toks = test_tokens[idx][:valid_len] if test_tokens[idx] else ["<UNK>"]

        test_attn_records.append({
            "idx": idx,
            "tokens": toks,
            "weights": weights,
            "true_emo": EMOTIONS[true_id],
            "pred_emo": EMOTIONS[pred_id],
            "confidence": conf,
            "correct": pred_id == true_id,
        })

        for tok, w in zip(toks, weights):
            if len(tok) > 1 or tok in {"!", "?"}:
                word_attn_by_true_emotion[EMOTIONS[true_id]][tok].append(float(w))

# Select 6 correct examples (one per emotion where possible) + 2 error examples
selected_samples = []
for emo in EMOTIONS:
    matches = [
        r for r in test_attn_records
        if r["correct"] and r["true_emo"] == emo and 4 <= len(r["tokens"]) <= 18
    ]
    if matches:
        selected_samples.append(matches[0])

errors = [
    r for r in test_attn_records
    if not r["correct"] and 4 <= len(r["tokens"]) <= 18
][:2]
selected_samples.extend(errors)
selected_samples = selected_samples[:8]

# Plot attention heatmaps for the 8 selected test sentences
fig, axes = plt.subplots(len(selected_samples), 1, figsize=(13, 2.1 * len(selected_samples)))
if len(selected_samples) == 1:
    axes = [axes]

for ax, sample in zip(axes, selected_samples):
    toks = sample["tokens"]
    w = sample["weights"][np.newaxis, :]
    im = ax.imshow(w, cmap="YlOrRd", aspect="auto", vmin=0.0, vmax=max(float(w.max()), 0.15))
    ax.set_xticks(range(len(toks)))
    ax.set_xticklabels(toks, rotation=25, ha="right", fontsize=10)
    ax.set_yticks([])
    status = "CORRECT" if sample["correct"] else "ERROR"
    ax.set_title(
        f"[{status}] True: {sample['true_emo']} | Pred: {sample['pred_emo']} (conf={sample['confidence']:.2f})",
        fontsize=11,
        fontweight="bold",
    )
    for j, val in enumerate(sample["weights"]):
        ax.text(j, 0, f"{val:.2f}", ha="center", va="center", fontsize=8, color="black")

plt.tight_layout()
plt.savefig("results/attention_heatmaps.png", dpi=150)
plt.show()

# Top-10 most attended words per emotion (min count = 5, fallback to min count = 2 in QUICK_MODE)
min_cnt = 2 if QUICK_MODE else 5
top_words_rows = []
for emo in EMOTIONS:
    scored = []
    for word, w_list in word_attn_by_true_emotion[emo].items():
        if len(w_list) >= min_cnt:
            scored.append((word, float(np.mean(w_list)), len(w_list)))
    scored.sort(key=lambda x: x[1], reverse=True)
    top10 = [f"{w} ({avg:.2f}, n={c})" for w, avg, c in scored[:10]]
    top_words_rows.append({
        "Emotion": emo,
        "Top_10_Attended_Words": ", ".join(top10) if top10 else "N/A (increase sample size)",
    })

top_words_df = pd.DataFrame(top_words_rows)
top_words_df.to_csv("results/top_attended_words.csv", index=False)
print("=== TOP-10 MOST ATTENDED WORDS PER EMOTION CLASS ===")
for row in top_words_rows:
    print(f"  {row['Emotion']:10s}: {row['Top_10_Attended_Words']}")

# Empirical discussion generated strictly from computed metrics
lstm_f1 = results["LSTM"]["macro_f1"]
gru_f1 = results["GRU"]["macro_f1"]
attn_f1 = results["BiLSTM+Attention"]["macro_f1"]
bert_f1 = results["DistilBERT"]["macro_f1"]

best_Model_test = max(model_names, key=lambda m: results[m]["macro_f1"])
best_emo_attn = max(EMOTIONS, key=lambda e: results["BiLSTM+Attention"]["report"][e]["f1-score"])
hardest_emo_attn = min(EMOTIONS, key=lambda e: results["BiLSTM+Attention"]["report"][e]["f1-score"])
hardest_emo_bert = min(EMOTIONS, key=lambda e: results["DistilBERT"]["report"][e]["f1-score"])

discussion = f"""
=== EMPIRICAL FINDINGS & ANALYSIS (COMPUTED FROM REAL RESULTS) ===
1. Architecture Progression:
   - Unidirectional LSTM achieved Test Macro F1 = {lstm_f1:.4f} (Accuracy = {results['LSTM']['accuracy']:.4f}),
     while GRU achieved Test Macro F1 = {gru_f1:.4f} (Accuracy = {results['GRU']['accuracy']:.4f}).
   - Adding Bidirectionality + Additive Attention (BiLSTM+Attention) reached Test Macro F1 = {attn_f1:.4f}
     (a delta of {attn_f1 - max(lstm_f1, gru_f1):+.4f} over the best unidirectional RNN), because masked additive
     attention directly weights salient emotion-bearing tokens rather than compressing the whole sequence into the final step.
   - Fine-tuned DistilBERT achieved Test Macro F1 = {bert_f1:.4f} and Test Accuracy = {results['DistilBERT']['accuracy']:.4f}.

2. Class Imbalance & Rare Classes:
   - In the training set, 'joy' has {train_counts[label2id['joy']]} examples ({dist_df.loc[dist_df['Emotion']=='joy', 'Train %'].values[0]:.1f}%),
     whereas rare emotions such as 'fear' ({train_counts[label2id['fear']]}) and 'disgust' ({train_counts[label2id['disgust']]}) have far fewer samples.
   - Even with inverse-frequency class weighting, the easiest class for BiLSTM+Attention is '{best_emo_attn}'
     (F1 = {results['BiLSTM+Attention']['report'][best_emo_attn]['f1-score']:.4f}), while the hardest class is '{hardest_emo_attn}'
     (F1 = {results['BiLSTM+Attention']['report'][hardest_emo_attn]['f1-score']:.4f}; for DistilBERT, hardest is '{hardest_emo_bert}'
     with F1 = {results['DistilBERT']['report'][hardest_emo_bert]['f1-score']:.4f}).
"""
print(discussion)

## Stage 10: Streamlit App (`app.py`) & One-Click Deployment Archive

First we write `app.py` using `%%writefile app.py`. The Streamlit app uses `@st.cache_resource` to call `load_predictor` once, supports optional Hugging Face Hub loading via `hub_repo`, displays emotion emojis and probability bar charts, and renders token-level attention heatmaps from the packaged `BiLSTM+Attention` model.

In [ ]:
%%writefile app.py
import html
import json
from pathlib import Path

import pandas as pd
import streamlit as st

from emotion_inference import EMOTIONS, load_predictor

st.set_page_config(page_title="Emotion Classifier with Attention", page_icon="😊", layout="centered")

EMOJI_MAP = {
    "joy": "😊",
    "sadness": "😢",
    "anger": "😠",
    "fear": "😨",
    "surprise": "😲",
    "disgust": "🤢",
}

DEFAULT_TEXT = "I am so grateful and thrilled that everything worked out today!"
EXAMPLES = [
    "I am so grateful and thrilled that everything worked out today!",
    "It breaks my heart how much I miss my old friends, I feel so lonely.",
    "Wait what?! I had no idea they were planning a surprise announcement!",
]

ROOT_DIR = Path(__file__).resolve().parent
CFG_PATH = ROOT_DIR / "deployment_config.json"

if not CFG_PATH.exists():
    st.error("Missing deployment_config.json in the application directory.")
    st.stop()

with open(CFG_PATH, "r", encoding="utf-8") as f:
    deploy_cfg = json.load(f)

best_model_name = deploy_cfg.get("best_model", "Unknown")
model_type = deploy_cfg.get("model_type", "rnn")
hub_repo = deploy_cfg.get("hub_repo")
model_dir = ROOT_DIR / "model"

if not (model_type == "transformer" and hub_repo):
    if not model_dir.exists():
        st.error("Local 'model/' folder is missing and no 'hub_repo' was configured in deployment_config.json.")
        st.stop()

@st.cache_resource
def get_cached_predictor():
    return load_predictor(ROOT_DIR)

predict_fn, cfg = get_cached_predictor()

# Sidebar with model info and metrics
with st.sidebar:
    st.header("Model Details")
    st.write(f"**Best Model:** `{best_model_name}`")
    st.write(f"**Architecture Type:** `{model_type}`")
    if hub_repo and model_type == "transformer":
        st.write(f"**HF Hub Repo:** `{hub_repo}`")
    metrics = cfg.get("metrics", {})
    if metrics:
        st.subheader("Evaluation Metrics")
        if "accuracy" in metrics:
            st.metric("Test Accuracy", f"{metrics['accuracy'] * 100:.2f}%")
        if "macro_f1" in metrics:
            st.metric("Test Macro F1", f"{metrics['macro_f1'] * 100:.2f}%")
        if "val_macro_f1" in metrics:
            st.metric("Validation Macro F1", f"{metrics['val_macro_f1'] * 100:.2f}%")

st.title("Emotion Classification with Attention")
st.caption(f"Powered by **{best_model_name}** (GoEmotions 6-Class Ekman Taxonomy)")

if "input_text" not in st.session_state:
    st.session_state["input_text"] = DEFAULT_TEXT

def set_example_text(text_val: str):
    st.session_state["input_text"] = text_val

st.write("**Try an example sentence:**")
cols = st.columns(3)
for idx, ex_text in enumerate(EXAMPLES):
    cols[idx].button(
        f"Example {idx + 1}",
        key=f"ex_btn_{idx}",
        on_click=set_example_text,
        args=(ex_text,),
    )

user_text = st.text_area(
    "Enter text to classify (press Ctrl+Enter to analyze):",
    key="input_text",
    max_chars=500,
    height=110,
)

if user_text and user_text.strip():
    result = predict_fn(user_text.strip())
    pred_label = result["label"]
    probs = result["probs"]
    conf = probs.get(pred_label, 0.0)
    emoji = EMOJI_MAP.get(pred_label, "✨")

    st.subheader("Prediction Result")
    st.markdown(f"### {emoji} **{pred_label.upper()}** — Confidence: `{conf * 100:.1f}%`")

    prob_df = pd.DataFrame(
        {"Probability": [probs.get(e, 0.0) for e in EMOTIONS]},
        index=EMOTIONS,
    )
    st.bar_chart(prob_df)

    st.subheader("Token Attention Heatmap")
    tokens = result.get("tokens") or []
    attn = result.get("attention")
    if tokens and attn:
        max_a = max(max(attn), 1e-6)
        spans = []
        for tok, weight in zip(tokens, attn):
            alpha = min(max((weight / max_a) * 0.85, 0.08), 0.90)
            safe_tok = html.escape(tok)
            spans.append(
                f'<span title="weight: {weight:.3f}" style="display:inline-block;margin:3px 4px;padding:4px 8px;'
                f'border-radius:6px;background-color:rgba(239,68,68,{alpha:.3f});color:#111827;'
                f'font-family:monospace;font-size:0.95rem;">{safe_tok} <small>({weight:.2f})</small></span>'
            )
        html_block = "<div style='line-height:2.0;padding:8px;border:1px solid #e5e7eb;border-radius:8px;'>" + "".join(spans) + "</div>"
        st.markdown(html_block, unsafe_allow_html=True)
    else:
        st.info("Attention weights are unavailable for this input.")
    st.caption("Note: Token-level attention weights are computed by the packaged BiLSTM+Attention model.")

In [ ]:
# Select best model strictly by VALIDATION macro F1
best_name = max(model_names, key=lambda m: results[m]["val_macro_f1"])
best_res = results[best_name]
best_is_transformer = best_name == "DistilBERT"

print(f"Selected Best Model by Validation Macro F1: {best_name} (Val Macro F1 = {best_res['val_macro_f1']:.4f})")

deploy_root = Path("emotion_deployment")
if deploy_root.exists():
    shutil.rmtree(deploy_root)

(deploy_root / "model").mkdir(parents=True, exist_ok=True)
(deploy_root / "results").mkdir(parents=True, exist_ok=True)

# Copy app.py, emotion_inference.py, and results/*
shutil.copy2("app.py", deploy_root / "app.py")
shutil.copy2("emotion_inference.py", deploy_root / "emotion_inference.py")
for item in Path("results").glob("*"):
    if item.is_file():
        shutil.copy2(item, deploy_root / "results" / item.name)

# Copy best model into emotion_deployment/model/
if best_is_transformer:
    shutil.copytree("models/distilbert-emotion", deploy_root / "model", dirs_exist_ok=True)
    rnn_cfg_for_json = results["BiLSTM+Attention"]["rnn_config"]
else:
    shutil.copy2(best_res["model_path"], deploy_root / "model" / "model.pt")
    shutil.copy2("models/vocab.json", deploy_root / "model" / "vocab.json")
    rnn_cfg_for_json = best_res["rnn_config"]

# Always package BiLSTM+Attention in attention_model/ unless BiLSTM+Attention is already the best model
if best_name != "BiLSTM+Attention":
    attn_dir = deploy_root / "attention_model"
    attn_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(results["BiLSTM+Attention"]["model_path"], attn_dir / "model.pt")
    shutil.copy2("models/vocab.json", attn_dir / "vocab.json")

# Write deployment_config.json
deployment_config = {
    "best_model": best_name,
    "model_type": "transformer" if best_is_transformer else "rnn",
    "labels": EMOTIONS,
    "rnn": rnn_cfg_for_json,
    "metrics": {
        "accuracy": round(best_res["accuracy"], 4),
        "macro_f1": round(best_res["macro_f1"], 4),
        "weighted_f1": round(best_res["weighted_f1"], 4),
        "val_macro_f1": round(best_res["val_macro_f1"], 4),
    },
    "all_models": overall_df.to_dict(orient="records"),
}
with open(deploy_root / "deployment_config.json", "w", encoding="utf-8") as f:
    json.dump(deployment_config, f, indent=2)

# Write requirements.txt
req_lines = [
    "--extra-index-url https://download.pytorch.org/whl/cpu",
    "streamlit>=1.38.0",
    "torch>=2.2.0",
    "numpy>=1.26.0",
    "pandas>=2.0.0",
]
if best_is_transformer:
    req_lines.append("transformers>=4.44.0")
with open(deploy_root / "requirements.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(req_lines) + "\n")

# Write README.md
readme_text = f"""# Emotion Classification with Attention — Deployment Package

- **Selected Best Model**: `{best_name}` (`model_type={deployment_config['model_type']}`)
- **Validation Macro F1**: `{best_res['val_macro_f1']:.4f}`
- **Test Macro F1**: `{best_res['macro_f1']:.4f}`
- **Test Accuracy**: `{best_res['accuracy']:.4f}`

## Quickstart (Local Deployment)

```bash
pip install -r requirements.txt
streamlit run app.py
```

## Package Contents
- `app.py`: Interactive Streamlit web application
- `emotion_inference.py`: Shared tokenizer, `SeqClassifier`, and `load_predictor`
- `deployment_config.json`: Model metadata and test evaluation metrics
- `model/`: Best model weights and tokenizer/vocabulary
- `attention_model/`: Packaged BiLSTM+Attention checkpoint for token attention visualization
- `results/`: Evaluation CSV tables and comparison/attention PNG plots
"""
with open(deploy_root / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_text)

# Zip into a single archive
zip_path = shutil.make_archive("emotion_deployment", "zip", ".", "emotion_deployment")
zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)

print(f"\nCreated archive: {zip_path} ({zip_size_mb:.2f} MB)")
print("Files inside emotion_deployment/:")
for p in sorted(deploy_root.rglob("*")):
    if p.is_file():
        sz_kb = p.stat().st_size / 1024
        print(f"  - {p.relative_to(deploy_root)} ({sz_kb:.1f} KB)")

In [ ]:
from IPython.display import HTML, display

verify_predict, verify_cfg = load_predictor("emotion_deployment")
print(f"Loaded deployment package | Best model: {verify_cfg['best_model']} ({verify_cfg['model_type']})")

sample_sentences = [
    "I am so grateful and thrilled that everything worked out today!",
    "It breaks my heart how much I miss my old friends, I feel so lonely.",
    "This is completely outrageous and unacceptable, stop lying to us!",
    "I am terrified and shaking right now, that sound in the dark scared me.",
    "That rotten food smelled absolutely disgusting and revolting.",
]

html_blocks = []
for s in sample_sentences:
    out = verify_predict(s)
    pred = out["label"]
    probs_str = ", ".join(f"{k}: {v:.2f}" for k, v in out["probs"].items())
    toks = out["tokens"]
    attn = out["attention"] or [0.0] * len(toks)
    max_w = max(max(attn), 1e-6)

    spans = []
    for tok, w in zip(toks, attn):
        alpha = min(max((w / max_w) * 0.85, 0.08), 0.90)
        spans.append(
            f"<span style='background:rgba(239,68,68,{alpha:.2f});padding:2px 6px;margin:2px;"
            f"border-radius:4px;font-family:monospace;'>{html.escape(tok)} ({w:.2f})</span>"
        )
    html_blocks.append(
        f"<div style='margin-bottom:12px;padding:10px;border:1px solid #cbd5e1;border-radius:8px;'>"
        f"<div><b>Input:</b> {html.escape(s)}</div>"
        f"<div><b>Predicted:</b> <span style='color:#0f172a;font-weight:bold;'>{pred.upper()}</span> "
        f"<small>[{probs_str}]</small></div>"
        f"<div style='margin-top:6px;'>{''.join(spans)}</div></div>"
    )

display(HTML("".join(html_blocks)))

### Downloading `emotion_deployment.zip`

The cell below first mounts **Google Drive** and copies `emotion_deployment.zip` to `MyDrive` (which is fast and reliable for large transformer archives), and then triggers `google.colab.files.download`.

> **Tip:** If the direct browser download hangs on a large zip file, open the **Files** panel on the left sidebar (folder icon) -> right-click `emotion_deployment.zip` -> **Download**, or download it directly from your **Google Drive** (`MyDrive/emotion_deployment.zip`).

In [ ]:
try:
    from google.colab import drive, files
    try:
        drive.mount("/content/drive", force_remount=False)
        drive_target = "/content/drive/MyDrive/emotion_deployment.zip"
        shutil.copy2("emotion_deployment.zip", drive_target)
        print(f"Copied archive to Google Drive: {drive_target}")
    except Exception as drive_err:
        print(f"Google Drive copy skipped ({drive_err}). Proceeding with direct download...")
    files.download("emotion_deployment.zip")
except ImportError:
    print("Not running inside Google Colab; archive is saved locally at ./emotion_deployment.zip")